In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import itertools
import random

# Get top N frequent numbers for each column position (N can be tweaked)
N = 10
df = pd.read_csv('Lottery/lottotexas.csv', header=None)

winning_number_columns = [4, 5, 6, 7, 8, 9]

print(df.head())    

             0   1   2     3   4   5   6   7   8   9
0  Lotto Texas  11  14  1992  13  16  22  29  32  36
1  Lotto Texas  11  18  1992  27  31  39  44  45  47
2  Lotto Texas  11  21  1992  11  21  24  28  31  46
3  Lotto Texas  11  25  1992  14  21  36  39  40  42
4  Lotto Texas  11  28  1992   9  17  21  24  28  50


In [8]:
#generate 5 unique combinations that never occurred in the dataset. 
#make sure to take most frequent (most probable) numbers for each of the last 6 columns into account

In [9]:

# Extract the last 6 columns (winning numbers: columns 4 to 9 inclusive)
winning_numbers = df.iloc[:, 4:10]

# Get all historical combinations as sorted tuples
existing_combos = set(winning_numbers.apply(lambda row: tuple(sorted(row)), axis=1))

# Get top N frequent numbers for each column position
N = 10  # You can adjust this to 12–15 if needed
top_numbers_per_position = []

for col in winning_numbers.columns:
    top_numbers = winning_numbers[col].value_counts().head(N).index.tolist()
    top_numbers_per_position.append(top_numbers)

# Generate candidate combinations using Cartesian product
candidate_combos = itertools.product(*top_numbers_per_position)

# Prepare set of unique new combinations
unique_new_combos = set()

for combo in candidate_combos:
    if len(set(combo)) < 6:
        continue  # skip if numbers repeat
    sorted_combo = tuple(sorted(combo))
    if sorted_combo in existing_combos:
        continue  # skip if already drawn
    unique_new_combos.add(sorted_combo)
    if len(unique_new_combos) == 5:
        break  # stop after finding 5 valid combos

# Print the results
print("5 new unique and probable 6-number combinations (no repeats within combo):\n")
for combo in unique_new_combos:
    print(combo)


5 new unique and probable 6-number combinations (no repeats within combo):

(4, 8, 21, 31, 42, 49)
(4, 8, 12, 21, 31, 42)
(4, 8, 21, 31, 37, 42)
(4, 8, 19, 21, 31, 42)
(4, 8, 16, 21, 31, 42)


In [12]:
from collections import defaultdict, Counter
import random

const COMBOS = 10

# Extract only the winning numbers (last 6 columns: columns 4 to 9)
draws = df.iloc[:, 4:10].reset_index(drop=True)
draws.columns = range(6)  # force column names to be 0, 1, ..., 5

# Get existing combinations as sorted tuples
existing_combos = set(draws.apply(lambda row: tuple(sorted(row)), axis=1))

# Step 1: Build correlation maps
correlation_maps = [defaultdict(Counter) for _ in range(5)]

for _, row in draws.iterrows():
    for i in range(5):  # positions 0 to 4
        current_num = row[i]
        next_num = row[i + 1]
        if current_num != next_num:
            correlation_maps[i][current_num][next_num] += 1

# Step 2: Choose starting points — most frequent first-position numbers
first_position_counts = draws[0].value_counts()
starting_candidates = first_position_counts.index.tolist()

# Step 3: Build new unique chains based on correlations
new_combinations = set()

def build_chain(start_num):
    chain = [start_num]
    used = set(chain)

    for i in range(5):
        current = chain[-1]
        next_candidates = correlation_maps[i].get(current, {})
        if not next_candidates:
            return None
        # Sort next candidates by frequency
        sorted_next = [n for n, _ in next_candidates.most_common() if n not in used]
        if not sorted_next:
            return None
        next_num = sorted_next[0]
        chain.append(next_num)
        used.add(next_num)

    return tuple(sorted(chain))  # normalize for duplicate check

# Step 4: Try multiple chains until we get 5 new ones
random.shuffle(starting_candidates)
for start in starting_candidates:
    combo = build_chain(start)
    if combo and combo not in existing_combos and combo not in new_combinations:
        new_combinations.add(combo)
    if len(new_combinations) == COMBOS:
        break

# Step 5: Output
print("10 new unique and probable 6-number combinations using number-wise correlation:\n")
for combo in new_combinations:
    print(combo)


5 new unique and probable 6-number combinations using number-wise correlation:

(29, 37, 43, 44, 48, 52)
(3, 24, 37, 38, 43, 44)
(14, 25, 26, 39, 42, 53)
(4, 5, 22, 37, 40, 49)
(22, 30, 33, 35, 37, 42)
(4, 7, 29, 43, 48, 54)
(1, 5, 14, 15, 35, 38)
(1, 5, 8, 15, 35, 38)
(10, 14, 25, 32, 42, 53)
(6, 7, 17, 21, 23, 26)


In [16]:
import pandas as pd
from collections import defaultdict, Counter
import random

# === CONFIGURATION ===
CSV_FILE = 'Lottery/lottotexas.csv'
NUM_COMBINATIONS = 10
TOP_N_PER_POSITION = 100  # Use top 10 frequent numbers per position for better candidate pool

# === STEP 1: Load data ===
df = pd.read_csv(CSV_FILE, header=None)
winning_numbers = df.iloc[:, 4:10].reset_index(drop=True)
winning_numbers.columns = range(6)  # Rename columns to 0–5

# === STEP 2: Build historical combination set (sorted for normalization) ===
existing_combos = set(winning_numbers.apply(lambda row: tuple(sorted(row)), axis=1))

# === STEP 3: Build frequency maps for each position (top N values) ===
top_numbers_per_position = []
for col in winning_numbers.columns:
    top_numbers = winning_numbers[col].value_counts().head(TOP_N_PER_POSITION).index.tolist()
    top_numbers_per_position.append(top_numbers)

# === STEP 4: Build position-wise correlation maps ===
# correlation_maps[i][number_at_pos_i] => Counter of most likely numbers at position i+1
correlation_maps = [defaultdict(Counter) for _ in range(5)]  # 0→1, 1→2, ..., 4→5

for _, row in winning_numbers.iterrows():
    for i in range(5):
        curr_num = row[i]
        next_num = row[i + 1]
        if curr_num != next_num:
            correlation_maps[i][curr_num][next_num] += 1

# === STEP 5: Generate candidate combinations using correlation chains ===
def build_chain(start_num):
    chain = [start_num]
    used = set(chain)

    for i in range(5):  # 5 transitions
        current = chain[-1]
        next_candidates = correlation_maps[i].get(current, {})
        if not next_candidates:
            return None
        sorted_next = [n for n, _ in next_candidates.most_common() if n not in used]
        if not sorted_next:
            return None
        next_num = sorted_next[0]
        chain.append(next_num)
        used.add(next_num)

    return tuple(sorted(chain))  # normalize for lookup

# === STEP 6: Generate combinations ===
new_combinations = set()
starting_numbers = top_numbers_per_position[0].copy()
random.shuffle(starting_numbers)

# Try generating until we reach the desired number
for start in starting_numbers:
    if len(new_combinations) >= NUM_COMBINATIONS:
        break
    combo = build_chain(start)
    if combo and combo not in existing_combos and combo not in new_combinations:
        new_combinations.add(combo)

# If not enough, try randomly sampling start numbers from top N again
if len(new_combinations) < NUM_COMBINATIONS:
    attempts = 0
    while len(new_combinations) < NUM_COMBINATIONS and attempts < 500:
        start = random.choice(top_numbers_per_position[0])
        combo = build_chain(start)
        if combo and combo not in existing_combos and combo not in new_combinations:
            new_combinations.add(combo)
        attempts += 1

# === STEP 7: Output ===
print(f"\n{len(new_combinations)} unique, probable combinations (based on frequency & correlation):\n")
for combo in new_combinations:
    print(combo)



10 unique, probable combinations (based on frequency & correlation):

(7, 17, 23, 26, 33, 46)
(1, 5, 10, 15, 16, 28)
(24, 30, 31, 42, 43, 46)
(15, 24, 29, 43, 44, 47)
(11, 14, 17, 20, 40, 41)
(5, 14, 24, 46, 48, 49)
(4, 5, 22, 33, 37, 49)
(12, 20, 31, 41, 42, 43)
(3, 16, 21, 26, 33, 37)
(3, 24, 37, 38, 43, 44)


In [17]:
NUM_COMBINATIONS = 10
TOP_N_PER_POSITION = 10  # use top 10 most frequent numbers per position

# === STEP 1: Load and prepare data ===
df = pd.read_csv(CSV_FILE, header=None)
winning_numbers = df.iloc[:, 4:10].reset_index(drop=True)
winning_numbers.columns = range(6)  # standardize column indices

# Build set of existing combinations (sorted tuples)
existing_combos = set(winning_numbers.apply(lambda row: tuple(sorted(row)), axis=1))

# === STEP 2: Get top N most frequent numbers per column ===
top_numbers_per_position = []
for col in winning_numbers.columns:
    top_n = winning_numbers[col].value_counts().head(TOP_N_PER_POSITION).index.tolist()
    top_numbers_per_position.append(top_n)

# === STEP 3: Build correlation maps ===
# correlation_maps[i][num_at_i] => Counter of most frequent num_at_(i+1)
correlation_maps = [defaultdict(Counter) for _ in range(5)]

for _, row in winning_numbers.iterrows():
    for i in range(5):  # positions 0 through 4
        current = row[i]
        next_val = row[i + 1]
        if current != next_val:
            correlation_maps[i][current][next_val] += 1

# === STEP 4: Build chain with fallback options ===
def build_chain(start_num):
    chain = [start_num]
    used = set(chain)

    for i in range(5):  # 5 transitions to build 6 numbers
        current = chain[-1]
        next_candidates = correlation_maps[i].get(current, {})
        if not next_candidates:
            return None
        # Try each candidate by frequency, skip already-used
        for next_num, _ in next_candidates.most_common():
            if next_num not in used:
                chain.append(next_num)
                used.add(next_num)
                break
        else:
            return None  # no valid next number found

    return tuple(sorted(chain))  # normalize for uniqueness check

# === STEP 5: Try generating combinations ===
new_combinations = set()

# Prioritize most frequent first-position numbers
first_pos_counts = winning_numbers[0].value_counts()
starting_numbers = first_pos_counts.index.tolist()

# Try to build chains from starting numbers
for start in starting_numbers:
    if len(new_combinations) >= NUM_COMBINATIONS:
        break
    combo = build_chain(start)
    if combo and combo not in existing_combos and combo not in new_combinations:
        new_combinations.add(combo)

# If not enough, retry randomly from top numbers
attempts = 0
while len(new_combinations) < NUM_COMBINATIONS and attempts < 500:
    start = random.choice(top_numbers_per_position[0])
    combo = build_chain(start)
    if combo and combo not in existing_combos and combo not in new_combinations:
        new_combinations.add(combo)
    attempts += 1

# === STEP 6: Output ===
print(f"\n✅ Generated {len(new_combinations)} new, unique 6-number combinations:\n")
for combo in new_combinations:
    print(combo)


✅ Generated 10 new, unique 6-number combinations:

(5, 14, 39, 46, 48, 49)
(14, 25, 26, 39, 42, 53)
(5, 18, 33, 42, 45, 47)
(1, 4, 5, 15, 16, 28)
(13, 20, 31, 41, 42, 43)
(1, 10, 12, 15, 31, 43)
(1, 2, 9, 18, 31, 49)
(1, 5, 14, 15, 35, 38)
(1, 5, 15, 16, 19, 28)
(10, 14, 25, 32, 42, 53)
